# Sidewalk Obstacle Footprint Mapping

Takes segmented masks of **sidewalks** and **obstacles** from a street-level image
and produces a **rectified bird's-eye view** showing ground-level obstacle footprints.

**Pipeline:**
1. Load sidewalk + obstacle masks
2. Detect sidewalk edges (excluding image borders)
3. Perspective-rectify to top-down view (horizontal + vertical via vanishing point)
4. Estimate ground-level footprints per obstacle
5. Generate top-view maps

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from skimage.measure import label, regionprops
from PIL import Image
import cv2
import os
import glob
import io
from dotenv import load_dotenv
from google.cloud import storage

## 2. Configuration

- **`IMAGE_NAME`** — blob path of the streetview image in GCS
- **`MASKS_PREFIX`** — GCS prefix for the segmentation results folder (e.g. `segmentation-results/0555_40.971156_29.0509826/forward/`); sidewalk and obstacle masks are auto-discovered from subfolders
- **Footprint parameters** — scan ratios, aspect ratio, max height

In [ ]:
# ── GCS credentials (from .env) ──
load_dotenv(os.path.join("..", ".env"))
GCS_BUCKET_NAME = os.environ.get("GCS_BUCKET_NAME", "")
GCP_PROJECT_ID  = os.environ.get("GCP_PROJECT_ID", "")

# ── Input ──
#IMAGE_NAME = "streetview/polygon_4v/20260221T142707Z/0549_forward_40.9728424_29.0459336_208.9.jpg"
IMAGE_NAME = "streetview/polygon_4v/20260329T023321Z-BAGDAT-30m/001-0006_forward_40.9715613_29.066227_305.6.jpg"

filename = os.path.basename(IMAGE_NAME)
img_id, direction, lat, lon, heading = filename.replace(".jpg","").split("_")

#MASKS_PREFIX = f"segmentation-results/{img_id}_{lat}_{lon}/{direction}"
MASKS_PREFIX = f"v2/segmentation-results/{img_id[4:]}_{lat}_{lon}/{direction}"


# ── Footprint estimation parameters ──
OBSTACLE_IS_TREE = {"tree"}            # types that use trunk-detection logic
FOOTPRINT_BASE_SCAN_RATIO = 0.15      # bottom 15% of non-tree obstacles for base width
TREE_TRUNK_SCAN_RATIO = 0.40          # bottom 40% of trees to find trunk
FOOTPRINT_ASPECT_RATIO = 1.0          # footprint height / width (1.0 = square)
FOOTPRINT_MAX_HEIGHT = 25             # max footprint height in pixels

# ── Scale ──
PIXELS_PER_METER = 1.0  # set to real value if calibrated; otherwise measurements are in pixels

In [ ]:
# ── Camera parameters ──
CAMERA_HEIGHT_M = 1.8     # metres
HFOV_DEG        = 90      # horizontal field of view
PITCH_DEG       = 0       # camera pitch (0 = horizontal)

## 3. Load Data

In [ ]:
# ── GCS helpers ──
def _gcs_bucket():
    client = storage.Client(project=GCP_PROJECT_ID)
    return client.bucket(GCS_BUCKET_NAME)

def download_image_from_gcs(blob_name: str) -> np.ndarray:
    """Download an image from GCS and return as numpy array."""
    blob = _gcs_bucket().blob(blob_name)
    data = blob.download_as_bytes()
    return np.array(Image.open(io.BytesIO(data)).convert("RGB"))

def download_mask_from_gcs(blob_name: str) -> np.ndarray:
    """Download a mask from GCS and return as boolean array."""
    blob = _gcs_bucket().blob(blob_name)
    data = blob.download_as_bytes()
    mask = np.array(Image.open(io.BytesIO(data)).convert("L"))
    return (mask > 127).astype(bool)

def list_blobs_under(prefix: str) -> list[str]:
    """List all blob names under a GCS prefix."""
    return [b.name for b in _gcs_bucket().list_blobs(prefix=prefix)]

# ── Fetch original image from GCS ──
print(f"Fetching image from GCS: {IMAGE_NAME}")
original_image = download_image_from_gcs(IMAGE_NAME)
print(f"  shape: {original_image.shape}")

# ── Auto-discover masks from GCS ──
# Structure: {MASKS_PREFIX}/{class}/mask_{idx}.png
print(f"\nListing masks under: {MASKS_PREFIX}/")
all_blobs = list_blobs_under(MASKS_PREFIX + "/")
mask_blobs = [b for b in all_blobs if b.endswith(".png")]

sidewalk_masks = {}      # {"sidewalk_000": mask, "sidewalk_001": mask, ...}
road_masks = {}          # {"road_000": mask, ...}
all_obstacle_masks = {}  # {"bollard_000": mask, "tree_001": mask, ...}

for blob_name in sorted(mask_blobs):
    parts = blob_name.split("/")
    class_name = parts[-2]    # "bollard", "sidewalk", "tree", etc.
    mask_file  = parts[-1]    # "mask_000.png"
    idx = mask_file.replace("mask_", "").replace(".png", "")

    print(f"  Downloading {class_name}/mask_{idx}.png ...", end=" ")
    mask = download_mask_from_gcs(blob_name)
    print(f"({mask.sum():,} px)")

    key = f"{class_name}_{idx}"
    if class_name == "sidewalk":
        sidewalk_masks[key] = mask
    elif class_name == "road":
        road_masks[key] = mask
    else:
        all_obstacle_masks[key] = mask

if not sidewalk_masks:
    raise FileNotFoundError(f"No sidewalk masks found under {MASKS_PREFIX}/sidewalk/")

# ── Combine all road masks into one ──
_ref_shape = next(iter(sidewalk_masks.values())).shape
combined_road_mask = np.zeros(_ref_shape, dtype=bool)
for rm in road_masks.values():
    combined_road_mask |= rm
print(f"\nRoad masks found: {len(road_masks)}  ({combined_road_mask.sum():,} px total)")

# ── Assign obstacles to sidewalk segments by proximity ──
# Obstacle masks don't share pixels with sidewalk masks (separate classes),
# so we match by horizontal proximity: find which sidewalk's column range
# is closest to each obstacle's horizontal center.
segments = {}  # {sw_key: {"sidewalk_mask": ..., "obstacle_masks": {...}}}

# Precompute each sidewalk's column range
sw_col_ranges = {}
for sw_key, sw_mask in sidewalk_masks.items():
    segments[sw_key] = {
        "sidewalk_mask": sw_mask,
        "obstacle_masks": {},
    }
    sw_cols = np.where(sw_mask.any(axis=0))[0]
    if len(sw_cols) > 0:
        sw_col_ranges[sw_key] = (sw_cols[0], sw_cols[-1])

for obs_key, obs_mask in all_obstacle_masks.items():
    obs_cols = np.where(obs_mask.any(axis=0))[0]
    if len(obs_cols) == 0:
        continue
    obs_center = (obs_cols[0] + obs_cols[-1]) / 2.0

    best_sw = None
    best_dist = float("inf")
    for sw_key, (c_lo, c_hi) in sw_col_ranges.items():
        # Distance = 0 if obstacle center is within sidewalk column range
        if c_lo <= obs_center <= c_hi:
            dist = 0.0
        else:
            dist = min(abs(obs_center - c_lo), abs(obs_center - c_hi))
        if dist < best_dist:
            best_dist = dist
            best_sw = sw_key

    if best_sw is not None:
        segments[best_sw]["obstacle_masks"][obs_key] = obs_mask

print(f"\n{'='*50}")
print(f"Found {len(sidewalk_masks)} sidewalk segment(s)")
for sw_key, seg in segments.items():
    n_obs = len(seg['obstacle_masks'])
    obs_list = ', '.join(seg['obstacle_masks'].keys()) if n_obs else '(none)'
    print(f"  {sw_key}: {seg['sidewalk_mask'].sum():,} px, {n_obs} obstacles → {obs_list}")

## 4. Helper Functions

In [ ]:
from sklearn.linear_model import RANSACRegressor, LinearRegression

BORDER_MARGIN = 3
RANSAC_RESIDUAL_THRESHOLD = 20.0  # px — edges further than this from consensus are outliers
RANSAC_MIN_SAMPLES = 0.3         # at least half the rows must agree
SHIFT_PERCENTILE = 10             # shift fitted line to 5th percentile of inliers for outer envelope
ROAD_TOUCH_MARGIN = 20            # px — sidewalk edge within this distance of road = confirmed edge
ROAD_CONFIRMED_WEIGHT = 5.0       # weight multiplier for road-confirmed rows in fitting

def find_row_edges(mask: np.ndarray, road_mask: np.ndarray = None, border_margin: int = BORDER_MARGIN,
                   ransac_threshold: float = RANSAC_RESIDUAL_THRESHOLD,
                   ransac_min_samples: float = RANSAC_MIN_SAMPLES,
                   road_touch_margin: int = ROAD_TOUCH_MARGIN):
    """
    For each row, find left and right sidewalk edges.
    
    Road-aware edge confirmation:
      If a sidewalk edge is within road_touch_margin of a road pixel,
      that edge is "road-confirmed" — guaranteed correct.
      Road-confirmed rows are forced inliers and given high weight
      in line fitting, since they represent the true sidewalk boundary.
    
    Left and right edges are validated independently.
    Uses RANSAC + directional inlier/outlier reclassification:
    - Left edge: points to the LEFT of fit are always inliers (can't be occlusion),
                 points to the RIGHT beyond threshold are always outliers (occlusion).
    - Right edge: mirrored logic.
    After reclassification, refits using only clean points.

    Returns (left_edges, right_edges, valid_mask, extrapolated_mask, edge_model).
    """
    # ── Determine which side of the road the sidewalk is on ──
    sw_center = np.mean(np.where(mask.any(axis=0))[0])
    if road_mask is not None and np.any(road_mask):
        road_center = np.mean(np.where(road_mask.any(axis=0))[0])
        is_left_sidewalk = sw_center < road_center
    else:
        is_left_sidewalk = None

    H, W = mask.shape
    left_edges = np.full(H, np.nan)
    right_edges = np.full(H, np.nan)
    valid = np.zeros(H, dtype=bool)
    extrapolated = np.zeros(H, dtype=bool)

    left_valid = np.zeros(H, dtype=bool)
    right_valid = np.zeros(H, dtype=bool)
    left_clipped = np.zeros(H, dtype=bool)
    right_clipped = np.zeros(H, dtype=bool)
    has_sidewalk = np.zeros(H, dtype=bool)

    # ── Road-confirmed tracking ──
    road_confirmed_L = np.zeros(H, dtype=bool)
    road_confirmed_R = np.zeros(H, dtype=bool)

    for r in range(H):
        cols = np.where(mask[r])[0]
        if len(cols) == 0:
            continue
        has_sidewalk[r] = True
        left_col = cols[0]
        right_col = cols[-1]

        left_edges[r] = left_col
        right_edges[r] = right_col

        left_at_border = left_col < border_margin
        right_at_border = right_col >= W - border_margin

        if not left_at_border:
            left_valid[r] = True
        else:
            left_clipped[r] = True

        if not right_at_border:
            right_valid[r] = True
        else:
            right_clipped[r] = True

        # ── Road proximity confirmation ──
        if road_mask is not None and is_left_sidewalk is not None:
            road_cols = np.where(road_mask[r])[0]
            if len(road_cols) > 0:
                if is_left_sidewalk:
                    # Sidewalk is LEFT of road → RIGHT edge is road-side
                    dist_right = np.min(np.abs(road_cols - right_col))
                    if dist_right <= road_touch_margin:
                        road_confirmed_R[r] = True
                        right_valid[r] = True   # override border-clip
                        right_clipped[r] = False
                    else:
                        # Road exists in this row but doesn't touch → suspect edge
                        right_valid[r] = False
                else:
                    # Sidewalk is RIGHT of road → LEFT edge is road-side
                    dist_left = np.min(np.abs(road_cols - left_col))
                    if dist_left <= road_touch_margin:
                        road_confirmed_L[r] = True
                        left_valid[r] = True    # override border-clip
                        left_clipped[r] = False
                    else:
                        left_valid[r] = False

    both_valid = left_valid & right_valid
    left_valid_idx = np.where(left_valid)[0]
    right_valid_idx = np.where(right_valid)[0]

    n_road_confirmed_L = road_confirmed_L[left_valid_idx].sum() if len(left_valid_idx) > 0 else 0
    n_road_confirmed_R = road_confirmed_R[right_valid_idx].sum() if len(right_valid_idx) > 0 else 0
    print(f"    Road-confirmed edges — left: {n_road_confirmed_L}, right: {n_road_confirmed_R}")

    edge_model = None

    if len(left_valid_idx) >= 2 and len(right_valid_idx) >= 2:

        # ══════════════════════════════════════════════════════
        # LEFT EDGE
        # ══════════════════════════════════════════════════════
        a_L, b_L, inlier_mask_L = _fit_edge(
            left_valid_idx, left_edges, road_confirmed_L,
            side="left",
            ransac_threshold=ransac_threshold,
            ransac_min_samples=ransac_min_samples,
        )

        # ══════════════════════════════════════════════════════
        # RIGHT EDGE
        # ══════════════════════════════════════════════════════
        a_R, b_R, inlier_mask_R = _fit_edge(
            right_valid_idx, right_edges, road_confirmed_R,
            side="right",
            ransac_threshold=ransac_threshold,
            ransac_min_samples=ransac_min_samples,
        )

        edge_model = {
            'a_L': a_L, 'b_L': b_L,
            'a_R': a_R, 'b_R': b_R,
            'inlier_mask_L': inlier_mask_L,
            'inlier_mask_R': inlier_mask_R,
            'left_valid_idx': left_valid_idx,
            'right_valid_idx': right_valid_idx,
            'valid_idx': np.where(both_valid)[0],
            'road_confirmed_L': road_confirmed_L,
            'road_confirmed_R': road_confirmed_R,
        }

        # Extrapolate clipped edges from the model
        for r in np.where(has_sidewalk)[0]:
            if left_clipped[r]:
                left_edges[r] = a_L * r + b_L
            if right_clipped[r]:
                right_edges[r] = a_R * r + b_R
            if left_clipped[r] or right_clipped[r]:
                extrapolated[r] = True

        valid = has_sidewalk.copy()

    else:
        valid = both_valid.copy()

    return left_edges, right_edges, valid, extrapolated, edge_model


def _fit_edge(valid_idx, edges, road_confirmed, side="left",
              ransac_threshold=RANSAC_RESIDUAL_THRESHOLD,
              ransac_min_samples=RANSAC_MIN_SAMPLES):
    """
    Fit a single edge (left or right) with road-confirmed priority.
    
    If enough road-confirmed rows exist (>= 4), fit primarily from those
    since they are guaranteed correct. Road-confirmed rows are always
    forced as inliers.
    
    side="left":  outliers are points pushed RIGHT  (residual > threshold)
    side="right": outliers are points pushed LEFT   (residual < -threshold)
    """
    rc_mask = road_confirmed[valid_idx]  # which valid rows are road-confirmed
    n_rc = rc_mask.sum()
    rc_rows = valid_idx[rc_mask]
    
    is_left = (side == "left")

    if n_rc >= 4:
        # ── Road-confirmed priority fit ──
        # Fit line from confirmed rows only (ground truth)
        a, b = np.polyfit(rc_rows.astype(float), edges[rc_rows], 1)
        
        # Envelope shift: use road-confirmed rows as reference
        fitted_rc = a * rc_rows.astype(float) + b
        residuals_rc = edges[rc_rows] - fitted_rc
        if is_left:
            shift = np.percentile(residuals_rc, SHIFT_PERCENTILE)
        else:
            shift = np.percentile(residuals_rc, 100 - SHIFT_PERCENTILE)
        b += shift

        # Classify all valid rows against this road-anchored line
        fitted_all = a * valid_idx.astype(float) + b
        residuals_all = edges[valid_idx] - fitted_all
        if is_left:
            inlier_mask = (residuals_all < ransac_threshold)
        else:
            inlier_mask = (residuals_all > -ransac_threshold)
        
        # Force road-confirmed rows as inliers (they are ground truth)
        inlier_mask[rc_mask] = True

        # Optional refit with all inliers (road-confirmed get high weight)
        if inlier_mask.sum() >= 2:
            clean_idx = valid_idx[inlier_mask]
            weights = np.ones(len(clean_idx), dtype=float)
            for i, r in enumerate(clean_idx):
                if road_confirmed[r]:
                    weights[i] = ROAD_CONFIRMED_WEIGHT
            # Weighted polyfit
            a, b = np.polyfit(clean_idx.astype(float), edges[clean_idx], 1, w=weights)
            
            # Re-apply envelope shift
            fitted2 = a * rc_rows.astype(float) + b
            residuals2 = edges[rc_rows] - fitted2
            if is_left:
                shift = np.percentile(residuals2, SHIFT_PERCENTILE)
            else:
                shift = np.percentile(residuals2, 100 - SHIFT_PERCENTILE)
            b += shift
            
            # Final inlier classification
            fitted3 = a * valid_idx.astype(float) + b
            residuals3 = edges[valid_idx] - fitted3
            if is_left:
                inlier_mask = (residuals3 < ransac_threshold)
            else:
                inlier_mask = (residuals3 > -ransac_threshold)
            inlier_mask[rc_mask] = True
        
        print(f"    {side.capitalize()} edge: road-confirmed fit ({n_rc} anchor rows, "
              f"{inlier_mask.sum()} total inliers)")
        return a, b, inlier_mask

    # ── Fallback: standard RANSAC (no/insufficient road confirmation) ──
    if len(valid_idx) >= 4:
        X = valid_idx.reshape(-1, 1).astype(float)
        
        # If we have SOME road-confirmed rows, use them as sample weight hints
        sample_weights = np.ones(len(valid_idx), dtype=float)
        if n_rc > 0:
            sample_weights[rc_mask] = ROAD_CONFIRMED_WEIGHT
        
        ransac = RANSACRegressor(
            estimator=LinearRegression(),
            residual_threshold=ransac_threshold,
            min_samples=ransac_min_samples,
            random_state=42
        )
        ransac.fit(X, edges[valid_idx], sample_weight=sample_weights)
        a = ransac.estimator_.coef_[0]
        b = ransac.estimator_.intercept_

        # Directional reclassification
        fitted = a * valid_idx.astype(float) + b
        residuals = edges[valid_idx] - fitted

        if is_left:
            inlier_mask = (residuals < ransac_threshold)
        else:
            inlier_mask = (residuals > -ransac_threshold)

        # Force road-confirmed as inliers
        if n_rc > 0:
            inlier_mask[rc_mask] = True

        if inlier_mask.sum() >= 2:
            clean_idx = valid_idx[inlier_mask]
            weights = np.ones(len(clean_idx), dtype=float)
            for i, r in enumerate(clean_idx):
                if road_confirmed[r]:
                    weights[i] = ROAD_CONFIRMED_WEIGHT
            a, b = np.polyfit(clean_idx.astype(float), edges[clean_idx], 1, w=weights)

            # Envelope shift
            fitted2 = a * clean_idx.astype(float) + b
            residuals_clean = edges[clean_idx] - fitted2
            if is_left:
                shift = np.percentile(residuals_clean, SHIFT_PERCENTILE)
            else:
                shift = np.percentile(residuals_clean, 100 - SHIFT_PERCENTILE)
            b += shift

            # Recompute inlier mask after shift
            fitted3 = a * valid_idx.astype(float) + b
            residuals3 = edges[valid_idx] - fitted3
            if is_left:
                inlier_mask = (residuals3 < ransac_threshold)
            else:
                inlier_mask = (residuals3 > -ransac_threshold)
            if n_rc > 0:
                inlier_mask[rc_mask] = True
    else:
        a, b = np.polyfit(valid_idx.astype(float), edges[valid_idx], 1)
        inlier_mask = np.ones(len(valid_idx), dtype=bool)

    return a, b, inlier_mask


def rectify_sidewalk(image_or_mask, left_edges, right_edges, valid_rows,
                     edge_model=None, target_width=None, is_mask=False,
                     cos_correction=1.0, f_px=None):
    """
    Per-row warp with horizontal AND vertical perspective correction.
    Samples along ground-plane perpendicular cross-sections using the
    perpendicular vanishing point for geometrically correct rectification.
    """
    H, W = image_or_mask.shape[:2]
    cy = H / 2.0

    valid_widths = right_edges[valid_rows] - left_edges[valid_rows]
    if target_width is None:
        target_width = int(np.median(valid_widths) * cos_correction)

    valid_idx = np.where(valid_rows)[0]
    all_rows = np.arange(H)

    left_interp = np.full(H, np.nan)
    right_interp = np.full(H, np.nan)

    if len(valid_idx) < 2:
        left_interp[:] = (left_edges[valid_idx[0]] if len(valid_idx) else 0)
        right_interp[:] = (right_edges[valid_idx[0]] if len(valid_idx) else W - 1)
        vy = -1e8
        first_valid = 0
        last_valid = H - 1
        a_L = 0.0
        b_L = left_interp[0]
        a_R = 0.0
        b_R = right_interp[0]
    else:
        if edge_model is not None:
            a_L, b_L = edge_model['a_L'], edge_model['b_L']
            a_R, b_R = edge_model['a_R'], edge_model['b_R']
        else:
            valid_r = valid_idx.astype(float)
            a_L, b_L = np.polyfit(valid_r, left_edges[valid_idx], 1)
            a_R, b_R = np.polyfit(valid_r, right_edges[valid_idx], 1)

        left_interp = a_L * all_rows + b_L
        right_interp = a_R * all_rows + b_R

        if abs(a_L - a_R) > 1e-6:
            vy = (b_R - b_L) / (a_L - a_R)
        else:
            vy = -1e8

        first_valid = valid_idx[0]
        last_valid = valid_idx[-1]

    # ── Perpendicular vanishing point ──
    use_perp = False
    vp_perp_x = 0.0
    if f_px is not None and abs(a_L - a_R) > 1e-6:
        vp_x = a_L * vy + b_L
        vp_x_offset = vp_x - W / 2.0
        
        if abs(vp_x_offset) > 1e-6:
            a_avg = (a_L + a_R) / 2.0
            sign = 1.0 if a_avg > 0 else -1.0
            vp_perp_x = W / 2.0 + sign * f_px * f_px / abs(vp_x_offset)
            use_perp = True

    # ── Vertical perspective correction ──
    row_scale = np.ones(H, dtype=np.float64)
    ref_dist = abs(last_valid - vy)
    MAX_STRETCH = 50.0

    for r in range(first_valid, last_valid + 1):
        dist = abs(r - vy)
        if dist > 0:
            s = (ref_dist / dist) ** 2
        else:
            s = MAX_STRETCH
        row_scale[r] = min(s, MAX_STRETCH)

    cum_real = np.cumsum(row_scale)
    cum_real = cum_real - cum_real[0]
    total_real_height = cum_real[-1]

    out_height = int(np.ceil(total_real_height)) + 1
    out_rows = np.arange(out_height, dtype=np.float32)
    src_row_for_out = np.interp(out_rows, cum_real, np.arange(H, dtype=np.float32))

    padding = int(target_width * 0.3)
    out_width = target_width + 2 * padding

    map_x = np.zeros((out_height, out_width), dtype=np.float32)
    map_y = np.zeros((out_height, out_width), dtype=np.float32)

    out_cols = np.arange(out_width, dtype=np.float32)

    for out_r in range(out_height):
        src_r = src_row_for_out[out_r]
        src_r_int = int(src_r)

        L = left_interp[min(src_r_int, H - 1)]
        R = right_interp[min(src_r_int, H - 1)]

        src_r_frac = src_r - src_r_int
        if src_r_frac > 0 and src_r_int + 1 < H:
            L_next = left_interp[src_r_int + 1]
            R_next = right_interp[src_r_int + 1]
            L = L * (1 - src_r_frac) + L_next * src_r_frac
            R = R * (1 - src_r_frac) + R_next * src_r_frac

        src_width = R - L
        if src_width <= 0:
            map_x[out_r, :] = -1
            map_y[out_r, :] = src_r
            continue

        cx = (L + R) / 2.0

        if use_perp and src_r > cy + 1:
            dx_perp = vp_perp_x - cx
            if abs(dx_perp) > 1e-6:
                perp_slope = (cy - src_r) / dx_perp

                denom_L = 1.0 - a_L * perp_slope
                denom_R = 1.0 - a_R * perp_slope

                if abs(denom_L) > 1e-9 and abs(denom_R) > 1e-9:
                    t_L = (L - cx) / denom_L
                    t_R = (R - cx) / denom_R
                    perp_span = t_R - t_L

                    if perp_span > 0:
                        t = t_L + (out_cols - padding) / target_width * perp_span
                        map_x[out_r, :] = cx + t
                        map_y[out_r, :] = src_r + t * perp_slope
                        continue

        # Fallback: horizontal sampling
        scale = src_width / target_width
        src_cols = L + (out_cols - padding) * scale
        map_x[out_r, :] = src_cols
        map_y[out_r, :] = src_r

    flags = cv2.INTER_NEAREST if is_mask else cv2.INTER_LINEAR
    inp = image_or_mask.astype(np.uint8) if is_mask else image_or_mask
    warped = cv2.remap(inp, map_x, map_y, interpolation=flags, borderMode=cv2.BORDER_CONSTANT)

    if is_mask:
        warped = warped.astype(bool)

    return warped, target_width, padding

def estimate_width_footprint(mask: np.ndarray, is_tree: bool = False,
                             base_scan_ratio: float = FOOTPRINT_BASE_SCAN_RATIO,
                             trunk_scan_ratio: float = TREE_TRUNK_SCAN_RATIO,
                             aspect_ratio: float = FOOTPRINT_ASPECT_RATIO,
                             max_height: int = FOOTPRINT_MAX_HEIGHT) -> np.ndarray:
    """
    Estimate the ground-level footprint of obstacles based on their width.
    Trees use trunk_scan_ratio, other obstacles use base_scan_ratio.
    """
    labeled, n = label(mask, return_num=True)
    footprint = np.zeros_like(mask)

    for reg in regionprops(labeled):
        min_row, min_col, max_row, max_col = reg.bbox
        height = max_row - min_row
        component = labeled == reg.label

        if height == 0:
            continue

        if is_tree:
            scan_rows = max(1, int(height * trunk_scan_ratio))
        else:
            scan_rows = max(3, int(height * base_scan_ratio))
        scan_rows = min(scan_rows, height)
        scan_start = max_row - scan_rows

        row_widths = []
        row_centers = []
        for r in range(scan_start, max_row):
            cols = np.where(component[r])[0]
            if len(cols) == 0:
                continue
            w = cols[-1] - cols[0] + 1
            row_widths.append(w)
            row_centers.append((cols[0] + cols[-1]) / 2.0)

        if not row_widths:
            continue

        fp_width = max(int(np.median(row_widths)), 1)
        median_center = np.median(row_centers)

        fp_height = max(1, int(fp_width * aspect_ratio))
        fp_height = min(fp_height, height, max_height)
        fp_top = max_row - fp_height
        fp_left = int(median_center - fp_width / 2.0)
        fp_right = fp_left + fp_width

        fp_left = max(0, fp_left)
        fp_right = min(mask.shape[1], fp_right)
        fp_top = max(0, fp_top)

        footprint[fp_top:max_row, fp_left:fp_right] = True

    return footprint


print("Functions defined: find_row_edges, _fit_edge, rectify_sidewalk, estimate_width_footprint")


In [ ]:
def visualize_rectification_geometry(original_image, sidewalk_mask, edge_model,
                                     f_px, HFOV_DEG, left_edges, right_edges,
                                     n_cross_sections=20):
    H, W = original_image.shape[:2]
    cy = H / 2.0

    a_L, b_L = edge_model['a_L'], edge_model['b_L']
    a_R, b_R = edge_model['a_R'], edge_model['b_R']
    left_valid_idx = edge_model['left_valid_idx']
    right_valid_idx = edge_model['right_valid_idx']
    inlier_L = edge_model['inlier_mask_L']
    inlier_R = edge_model['inlier_mask_R']
    road_confirmed_L = edge_model.get('road_confirmed_L', np.zeros(original_image.shape[0], dtype=bool))
    road_confirmed_R = edge_model.get('road_confirmed_R', np.zeros(original_image.shape[0], dtype=bool))

    if abs(a_L - a_R) > 1e-6:
        vy = (b_R - b_L) / (a_L - a_R)
        vp_x = a_L * vy + b_L
    else:
        vy = -1e8
        vp_x = W / 2.0

    vp_x_offset = vp_x - W / 2.0
    if abs(vp_x_offset) > 1e-6:
        a_avg = (a_L + a_R) / 2.0
        sign = 1.0 if a_avg > 0 else -1.0
        vp_perp_x = W / 2.0 + sign * f_px * f_px / abs(vp_x_offset)
    else:
        vp_perp_x = None

    fig, axes = plt.subplots(1, 3, figsize=(24, 10))

    # ── Panel 1: Edge fitting with correct indexing ──
    ax = axes[0]
    ax.imshow(original_image, alpha=0.5)
    sw_overlay = np.zeros((*sidewalk_mask.shape, 4))
    sw_overlay[sidewalk_mask] = [0.2, 0.6, 1.0, 0.2]
    ax.imshow(sw_overlay)

    # Plot LEFT edge points (indexed by left_valid_idx)
    for i, row in enumerate(left_valid_idx):
        if road_confirmed_L[row]:
            ax.plot(left_edges[row], row, 'D', color='cyan', markersize=3, zorder=6)
        elif i < len(inlier_L) and inlier_L[i]:
            ax.plot(left_edges[row], row, '.', color='lime', markersize=2)
        else:
            ax.plot(left_edges[row], row, 'x', color='red', markersize=4)

    # Plot RIGHT edge points (indexed by right_valid_idx)
    for i, row in enumerate(right_valid_idx):
        if road_confirmed_R[row]:
            ax.plot(right_edges[row], row, 'D', color='cyan', markersize=3, zorder=6)
        elif i < len(inlier_R) and inlier_R[i]:
            ax.plot(right_edges[row], row, '.', color='yellow', markersize=2)
        else:
            ax.plot(right_edges[row], row, 'x', color='orange', markersize=4)

    # Plot fitted lines
    plot_rows = np.array([0, H - 1])
    ax.plot(a_L * plot_rows + b_L, plot_rows, 'lime', linewidth=2, linestyle='--',
            label='Fitted left edge')
    ax.plot(a_R * plot_rows + b_R, plot_rows, 'yellow', linewidth=2, linestyle='--',
            label='Fitted right edge')

    if 0 <= vp_x <= W and -H < vy < H:
        ax.plot(vp_x, vy, '*', color='cyan', markersize=15, zorder=10,
                label=f'VP ({vp_x:.0f}, {vy:.0f})')

    ax.plot([], [], 'D', color='cyan', markersize=6, label='Road-confirmed')
    ax.plot([], [], '.', color='lime', markersize=8, label='Left inliers')
    ax.plot([], [], 'x', color='red', markersize=8, label='Left outliers')
    ax.plot([], [], '.', color='yellow', markersize=8, label='Right inliers')
    ax.plot([], [], 'x', color='orange', markersize=8, label='Right outliers')

    n_out_L = (~inlier_L).sum() if len(inlier_L) > 0 else 0
    n_out_R = (~inlier_R).sum() if len(inlier_R) > 0 else 0
    n_rc_L = road_confirmed_L[left_valid_idx].sum()
    n_rc_R = road_confirmed_R[right_valid_idx].sum()
    rc_info = f" | Road-confirmed: L={n_rc_L}, R={n_rc_R}" if (n_rc_L + n_rc_R) > 0 else ""
    ax.set_title(f"Edge Fitting\n"
                 f"Left: {inlier_L.sum()} inliers, {n_out_L} outliers | "
                 f"Right: {inlier_R.sum()} inliers, {n_out_R} outliers{rc_info}",
                 fontsize=11)
    ax.legend(loc='upper right', fontsize=7)
    ax.axis('off')

    # ── Panel 2: Horizontal rows vs perpendicular cross-sections ──
    ax = axes[1]
    ax.imshow(original_image, alpha=0.5)
    ax.imshow(sw_overlay)

    ax.plot(a_L * plot_rows + b_L, plot_rows, 'lime', linewidth=1.5, alpha=0.5)
    ax.plot(a_R * plot_rows + b_R, plot_rows, 'yellow', linewidth=1.5, alpha=0.5)

    sw_rows = np.where(sidewalk_mask.any(axis=1))[0]
    sw_rows = sw_rows[sw_rows > cy + 5]
    if len(sw_rows) == 0:
        sw_rows = np.where(sidewalk_mask.any(axis=1))[0]

    sample_rows = sw_rows[np.linspace(0, len(sw_rows) - 1, n_cross_sections, dtype=int)]

    for i, r in enumerate(sample_rows):
        L = a_L * r + b_L
        R = a_R * r + b_R
        cx = (L + R) / 2.0

        ax.plot([L, R], [r, r], color='red', linewidth=1.0, linestyle='--', alpha=0.7)

        if vp_perp_x is not None and r > cy + 1:
            dx_perp = vp_perp_x - cx
            if abs(dx_perp) > 1e-6:
                perp_slope = (cy - r) / dx_perp
                denom_L = 1.0 - a_L * perp_slope
                denom_R = 1.0 - a_R * perp_slope
                if abs(denom_L) > 1e-9 and abs(denom_R) > 1e-9:
                    t_L = (L - cx) / denom_L
                    t_R = (R - cx) / denom_R
                    x1 = cx + t_L
                    y1 = r + t_L * perp_slope
                    x2 = cx + t_R
                    y2 = r + t_R * perp_slope
                    ax.plot([x1, x2], [y1, y2], color='cyan', linewidth=1.5, alpha=0.8)

    ax.plot([], [], color='red', linewidth=2, linestyle='--', label='Horizontal rows')
    ax.plot([], [], color='cyan', linewidth=2, label='Perpendicular (true sidewalk rows)')
    ax.set_title(f"Horizontal Rows vs Perpendicular Cross-Sections\n"
                 f"({n_cross_sections} samples)", fontsize=11)
    ax.legend(loc='upper right', fontsize=8)
    ax.axis('off')

    # ── Panel 3: Geometry diagram ──
    ax = axes[2]
    ax.imshow(original_image, alpha=0.3)

    ax.plot(a_L * plot_rows + b_L, plot_rows, 'lime', linewidth=2, label='Left edge')
    ax.plot(a_R * plot_rows + b_R, plot_rows, 'yellow', linewidth=2, label='Right edge')

    ax.axhline(y=cy, color='white', linewidth=1, linestyle=':', alpha=0.5, label='Horizon')

    if -2 * H < vy < 2 * H:
        ax.plot(vp_x, vy, '*', color='cyan', markersize=20, zorder=10,
                label=f'Sidewalk VP ({vp_x:.0f}, {vy:.0f})')

    if vp_perp_x is not None:
        vp_perp_x_clipped = np.clip(vp_perp_x, -W, 2 * W)
        ax.plot(vp_perp_x_clipped, cy, 'D', color='magenta', markersize=12, zorder=10,
                label=f'Perp VP ({vp_perp_x:.0f}, {cy:.0f})')

        for r in sample_rows[::3]:
            L = a_L * r + b_L
            R = a_R * r + b_R
            cx_r = (L + R) / 2.0
            dx = vp_perp_x_clipped - cx_r
            dy = cy - r
            length = np.sqrt(dx**2 + dy**2)
            if length > 0:
                ux, uy = dx / length, dy / length
                ext = min(300, length * 0.3)
                ax.annotate('', xy=(cx_r + ux * ext, r + uy * ext),
                           xytext=(cx_r - ux * ext, r - uy * ext),
                           arrowprops=dict(arrowstyle='->', color='magenta',
                                         lw=1, alpha=0.4))

    a_C = (a_L + a_R) / 2.0
    b_C = (b_L + b_R) / 2.0
    ax.plot(a_C * plot_rows + b_C, plot_rows, 'white', linewidth=1, linestyle='-.',
            alpha=0.6, label='Centerline')

    ground_angle = np.degrees(np.arctan((vp_x - W / 2.0) / f_px))
    ax.set_title(f"Geometry Diagram\n"
                 f"Ground angle: {ground_angle:.1f}° | "
                 f"cos correction: {np.cos(np.radians(ground_angle)):.3f}",
                 fontsize=11)
    ax.legend(loc='upper right', fontsize=7)
    ax.set_xlim(-W * 0.1, W * 1.1)
    ax.set_ylim(H * 1.1, -H * 0.3)
    ax.axis('off')

    plt.suptitle("Rectification Geometry Diagnostic", fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

    print(f"Sidewalk VP: ({vp_x:.1f}, {vy:.1f})")
    if vp_perp_x is not None:
        print(f"Perpendicular VP: ({vp_perp_x:.1f}, {cy:.1f})")
    print(f"Ground angle: {ground_angle:.1f}°")
    print(f"Left edge: y = {a_L:.4f} * row + {b_L:.1f}")
    print(f"Right edge: y = {a_R:.4f} * row + {b_R:.1f}")
    print(f"Outliers: left={n_out_L}, right={n_out_R}")

## 5. Run Pipeline Per Sidewalk Segment

For each sidewalk mask segment, independently:
1. Find sidewalk edges → detect vanishing point
2. Perspective-rectify all masks (horizontal + vertical)
3. Estimate ground-level obstacle footprints
4. Generate top-view maps

In [ ]:
from matplotlib.patches import ConnectionPatch

for seg_idx, (sw_key, seg) in enumerate(segments.items()):
    sidewalk_mask = seg["sidewalk_mask"]
    obstacle_masks_full = seg["obstacle_masks"]
    n_obs = len(obstacle_masks_full)

    print(f"\n{'='*60}")
    print(f"SEGMENT {seg_idx+1}/{len(segments)}: {sw_key}  ({n_obs} obstacles)")
    print(f"{'='*60}")

    # ── Color palette ──
    OBSTACLE_COLORS = {}
    _cmap = plt.cm.get_cmap("tab10", max(n_obs, 1))
    for i, obs_type in enumerate(obstacle_masks_full):
        OBSTACLE_COLORS[obs_type] = _cmap(i)[:3]

    # ── Combined masks (pre-rectification) ──
    obstacle_mask = np.zeros_like(sidewalk_mask, dtype=bool)
    for m in obstacle_masks_full.values():
        obstacle_mask |= m
    effective_mask = sidewalk_mask & ~obstacle_mask

    # ══════════════════════════════════════════════════════════
    # A. Input Visualization
    # ══════════════════════════════════════════════════════════
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(original_image)
    axes[0].set_title("Original Image")

    axes[1].imshow(sidewalk_mask, cmap="Blues")
    axes[1].set_title(f"Sidewalk Mask ({sw_key})")

    overlay = np.zeros((*sidewalk_mask.shape, 3), dtype=np.float32)
    overlay[effective_mask] = [0.2, 0.6, 1.0]
    for obs_type in obstacle_masks_full:
        overlay[obstacle_masks_full[obs_type]] = OBSTACLE_COLORS[obs_type]

    handles = [Patch(facecolor=(0.2, 0.6, 1.0), label="Usable sidewalk")]
    for obs_type in obstacle_masks_full:
        handles.append(Patch(facecolor=OBSTACLE_COLORS[obs_type], label=obs_type))

    axes[2].imshow(overlay)
    axes[2].set_title("Obstacle Silhouettes")
    if handles:
        axes[2].legend(handles=handles, loc="upper right", fontsize=8)

    for ax in axes:
        ax.axis("off")
    plt.suptitle(f"Segment: {sw_key}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════
    # B. Edge Detection
    # ══════════════════════════════════════════════════════════
    left_edges, right_edges, valid_rows, extrapolated_rows, edge_model = find_row_edges(sidewalk_mask, road_mask=combined_road_mask)
    n_valid = valid_rows.sum()
    n_extrapolated = extrapolated_rows.sum()
    n_measured = n_valid - n_extrapolated  # rows with real observed edges
    n_total = sidewalk_mask.any(axis=1).sum()

    print(f"Rows with sidewalk: {n_total}")
    print(f"Rows with valid (non-border) edges: {n_measured}")
    print(f"Rows extrapolated from model: {n_extrapolated}")
    print(f"Rows skipped (no sidewalk): {sidewalk_mask.shape[0] - n_total}")

    if edge_model is not None:
        n_left_valid = len(edge_model['left_valid_idx'])
        n_right_valid = len(edge_model['right_valid_idx'])
        n_inliers_L = edge_model['inlier_mask_L'].sum()
        n_inliers_R = edge_model['inlier_mask_R'].sum()
        print(f"Left edge:  {n_left_valid} valid rows, RANSAC kept {n_inliers_L}")
        print(f"Right edge: {n_right_valid} valid rows, RANSAC kept {n_inliers_R}")

    if n_valid == 0:
        print(f"⚠ No valid edges found for {sw_key} — skipping.")
        continue

    # Separate measured vs extrapolated width stats
    measured_rows = valid_rows & ~extrapolated_rows
    if measured_rows.any():
        widths_measured = right_edges[measured_rows] - left_edges[measured_rows]
        print(f"Measured sidewalk width range: {widths_measured.min():.0f} — {widths_measured.max():.0f} px")
    if extrapolated_rows.any():
        widths_extrap = right_edges[extrapolated_rows] - left_edges[extrapolated_rows]
        print(f"Extrapolated width range: {widths_extrap.min():.0f} — {widths_extrap.max():.0f} px  (estimated)")

    # Ground-plane angle correction (reused by rectification and measurement)
    H_img, W_img = original_image.shape[:2]
    f_px = W_img / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
    if edge_model is not None and abs(edge_model['a_L'] - edge_model['a_R']) > 1e-6:
        vp_y = (edge_model['b_R'] - edge_model['b_L']) / (edge_model['a_L'] - edge_model['a_R'])
        vp_x = edge_model['a_L'] * vp_y + edge_model['b_L']
        ground_angle = np.arctan((vp_x - W_img / 2.0) / f_px)
    else:
        ground_angle = 0.0
    cos_correction = np.cos(ground_angle)
    if cos_correction < 1.0:
        print(f"Ground angle: {np.degrees(ground_angle):.1f}°, cos correction: {cos_correction:.3f}")

    # Edge visualization
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.imshow(original_image, alpha=0.6)
    sw_overlay = np.zeros((*sidewalk_mask.shape, 4))
    sw_overlay[sidewalk_mask] = [0.2, 0.6, 1.0, 0.3]
    ax.imshow(sw_overlay)

    valid_row_indices = np.where(valid_rows)[0]
    ax.plot(left_edges[valid_rows], valid_row_indices, color="lime", linewidth=1.5, label="Left edge")
    ax.plot(right_edges[valid_rows], valid_row_indices, color="yellow", linewidth=1.5, label="Right edge")

    all_sw_rows = np.where(sidewalk_mask.any(axis=1))[0]
    skipped_rows = np.setdiff1d(all_sw_rows, valid_row_indices)
    if len(skipped_rows) > 0:
        for r in skipped_rows:
            cols = np.where(sidewalk_mask[r])[0]
            if len(cols) > 0:
                ax.plot([cols[0], cols[-1]], [r, r], color="red", linewidth=0.5, alpha=0.4)
        ax.plot([], [], color="red", linewidth=2, alpha=0.4, label=f"Skipped ({len(skipped_rows)})")

    ax.set_title(f"Detected Edges — {sw_key}")
    ax.legend(loc="upper right")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
    
    visualize_rectification_geometry(original_image, sidewalk_mask, edge_model,
                                     f_px, HFOV_DEG, left_edges, right_edges,
                                     n_cross_sections=25)
    # ══════════════════════════════════════════════════════════
    # C. Rectification + Footprint Estimation
    # ══════════════════════════════════════════════════════════
    sidewalk_mask_rect, target_w, pad = rectify_sidewalk(
        sidewalk_mask, left_edges, right_edges, valid_rows,
        edge_model=edge_model, is_mask=True, cos_correction=cos_correction, f_px=f_px)

    obstacle_masks_full_rect = {}
    for obs_type in obstacle_masks_full:
        full_rect, _, _ = rectify_sidewalk(
            obstacle_masks_full[obs_type], left_edges, right_edges, valid_rows,
            edge_model=edge_model, target_width=target_w, is_mask=True, f_px=f_px)
        obstacle_masks_full_rect[obs_type] = full_rect

    obstacle_masks_rect = {}
    for obs_type, full_rect in obstacle_masks_full_rect.items():
        is_tree = any(t in obs_type for t in OBSTACLE_IS_TREE)
        fp = estimate_width_footprint(full_rect, is_tree=is_tree)
        obstacle_masks_rect[obs_type] = fp

        method = "trunk-width" if is_tree else "base-width"
        reduction = (1 - fp.sum() / max(full_rect.sum(), 1)) * 100
        print(f"  '{obs_type}' footprint ({method}): {fp.sum():,} px  "
              f"({reduction:.0f}% reduction)")

    obstacle_mask_rect = np.zeros_like(sidewalk_mask_rect, dtype=bool)
    for m in obstacle_masks_rect.values():
        obstacle_mask_rect |= m
    effective_mask_rect = sidewalk_mask_rect & ~obstacle_mask_rect

    original_image_rect, _, _ = rectify_sidewalk(
        original_image, left_edges, right_edges, valid_rows,
        edge_model=edge_model, target_width=target_w, f_px=f_px)

    print(f"Rectified size: {sidewalk_mask_rect.shape[1]} × {sidewalk_mask_rect.shape[0]} px")

    # ══════════════════════════════════════════════════════════
    # D. Before vs After Comparison
    # ══════════════════════════════════════════════════════════
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))

    # Top: original perspective
    axes[0, 0].imshow(original_image)
    axes[0, 1].imshow(sidewalk_mask, cmap="Blues")
    overlay_pre = np.zeros((*sidewalk_mask.shape, 3), dtype=np.float32)
    overlay_pre[effective_mask] = [0.2, 0.6, 1.0]
    for ot in obstacle_masks_full:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        overlay_pre[obstacle_masks_full[ot]] = c
    axes[0, 2].imshow(overlay_pre)

    # Bottom: rectified
    axes[1, 0].imshow(original_image_rect)
    axes[1, 1].imshow(sidewalk_mask_rect, cmap="Blues")

    overlay_rect = np.zeros((*sidewalk_mask_rect.shape, 3), dtype=np.float32)
    overlay_rect[effective_mask_rect] = [0.2, 0.6, 1.0]
    for ot in obstacle_masks_full_rect:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        overlay_rect[obstacle_masks_full_rect[ot]] = [c[0]*0.4, c[1]*0.4, c[2]*0.4]
    for ot in obstacle_masks_rect:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        overlay_rect[obstacle_masks_rect[ot]] = c
    axes[1, 2].imshow(overlay_rect)

    left_col = pad
    right_col = pad + target_w
    for ax_idx in range(3):
        axes[1, ax_idx].axvline(x=left_col, color="lime", linewidth=2, linestyle="--", alpha=0.7)
        axes[1, ax_idx].axvline(x=right_col, color="yellow", linewidth=2, linestyle="--", alpha=0.7)

    titles_top = ["Original Image", "Sidewalk Mask", "Silhouettes"]
    titles_bot = ["Rectified Image", "Rectified Sidewalk", "Footprints (rectified)"]
    for i in range(3):
        axes[0, i].set_title(titles_top[i]); axes[0, i].axis("off")
        axes[1, i].set_title(titles_bot[i]); axes[1, i].axis("off")

    plt.suptitle(f"Before vs After Rectification — {sw_key}", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════
    # E. Top-View Maps
    # ══════════════════════════════════════════════════════════
    sw = sidewalk_mask_rect
    eff = effective_mask_rect

    fig, axes = plt.subplots(1, 3, figsize=(20, 10))

    # Panel 1: footprints
    tv = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv[sw] = [0.85, 0.85, 0.85]
    tv[eff] = [0.75, 0.9, 1.0]
    for ot, fp in obstacle_masks_rect.items():
        tv[fp] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
    axes[0].imshow(tv)
    axes[0].axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    axes[0].axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)
    axes[0].set_title("Ground Footprints", fontsize=13, fontweight="bold")
    axes[0].axis("off")

    legend_handles = [
        Patch(facecolor=(0.75, 0.9, 1.0), edgecolor="gray", label="Usable sidewalk"),
        Patch(facecolor=(0.85, 0.85, 0.85), edgecolor="gray", label="Sidewalk (blocked)"),
    ]
    for ot in obstacle_masks_rect:
        is_tree = any(t in ot for t in OBSTACLE_IS_TREE)
        method = "trunk" if is_tree else "base"
        legend_handles.append(Patch(facecolor=OBSTACLE_COLORS[ot], label=f"{ot} ({method})"))
    axes[0].legend(handles=legend_handles, loc="upper right", fontsize=8,
                   framealpha=0.9, facecolor="white")

    # Panel 2: full silhouettes
    tv2 = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv2[sw] = [0.85, 0.85, 0.85]
    for ot, full in obstacle_masks_full_rect.items():
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        tv2[full] = [c[0]*0.6, c[1]*0.6, c[2]*0.6]
    axes[1].imshow(tv2)
    axes[1].axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    axes[1].axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)
    axes[1].set_title("Full Silhouettes (reference)", fontsize=13)
    axes[1].axis("off")

    # Panel 3: overlay
    tv3 = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv3[sw] = [0.85, 0.85, 0.85]
    tv3[eff] = [0.75, 0.9, 1.0]
    for ot in obstacle_masks_rect:
        c = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        if ot in obstacle_masks_full_rect:
            tv3[obstacle_masks_full_rect[ot]] = [c[0]*0.3, c[1]*0.3, c[2]*0.3]
        tv3[obstacle_masks_rect[ot]] = c
    axes[2].imshow(tv3)
    axes[2].axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    axes[2].axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)
    axes[2].set_title("Footprint + Silhouette Overlay", fontsize=13)
    axes[2].axis("off")

    plt.suptitle(f"Top View — {sw_key}", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    # ── Footprint-only view ──
    fig, ax = plt.subplots(figsize=(8, 12))
    tv_fp = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv_fp[sw] = [0.75, 0.9, 1.0]
    for ot, full in obstacle_masks_full_rect.items():
        fp = obstacle_masks_rect.get(ot, np.zeros_like(full))
        above = full & ~fp
        tv_fp[above] = [0.75, 0.9, 1.0]
    for ot, fp in obstacle_masks_rect.items():
        tv_fp[fp] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))

    ax.imshow(tv_fp)
    ax.axvline(x=pad, color="white", linewidth=1.5, alpha=0.6)
    ax.axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.6)

    lh = [Patch(facecolor=(0.75, 0.9, 1.0), edgecolor="gray", label="Sidewalk")]
    for ot in obstacle_masks_rect:
        is_tree = any(t in ot for t in OBSTACLE_IS_TREE)
        method = "trunk" if is_tree else "base"
        lh.append(Patch(facecolor=OBSTACLE_COLORS[ot], label=f"{ot} ({method})"))
    ax.legend(handles=lh, loc="upper right", fontsize=9, framealpha=0.9, facecolor="white")

    ax.set_title(f"Footprint Only — {sw_key}", fontsize=14, fontweight="bold")
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    # ══════════════════════════════════════════════════════════
    # F. Silhouette → Footprint Mapping with Width Annotations
    # ══════════════════════════════════════════════════════════
    fig, (ax_sil, ax_fp) = plt.subplots(1, 2, figsize=(18, 12))

    # Left panel: full rectified silhouettes
    tv_left = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv_left[sw] = [0.85, 0.85, 0.85]
    tv_left[eff] = [0.75, 0.9, 1.0]
    for ot, full in obstacle_masks_full_rect.items():
        tv_left[full] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
    ax_sil.imshow(tv_left)
    ax_sil.axvline(x=pad, color="white", linewidth=1, alpha=0.4)
    ax_sil.axvline(x=pad + target_w, color="white", linewidth=1, alpha=0.4)
    ax_sil.set_title("Rectified Silhouettes", fontsize=13, fontweight="bold")
    ax_sil.axis("off")

    # Right panel: ground footprints
    tv_right = np.ones((*sw.shape, 3), dtype=np.float32) * 0.15
    tv_right[sw] = [0.85, 0.85, 0.85]
    tv_right[eff] = [0.75, 0.9, 1.0]
    for ot, fp in obstacle_masks_rect.items():
        tv_right[fp] = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
    ax_fp.imshow(tv_right)
    ax_fp.axvline(x=pad, color="white", linewidth=1, alpha=0.4)
    ax_fp.axvline(x=pad + target_w, color="white", linewidth=1, alpha=0.4)
    ax_fp.set_title("Ground Footprints", fontsize=13, fontweight="bold")
    ax_fp.axis("off")

    # Sidewalk width annotation at a few sample rows
    sw_rows_with_data = np.where(sw.any(axis=1))[0]
    if len(sw_rows_with_data) > 2:
        sample_positions = [0.25, 0.5, 0.75]
        for frac in sample_positions:
            sample_row = sw_rows_with_data[int(len(sw_rows_with_data) * frac)]
            sw_cols_at_row = np.where(sw[sample_row])[0]
            if len(sw_cols_at_row) > 0:
                sw_left = sw_cols_at_row[0]
                sw_right = sw_cols_at_row[-1]
                sw_w = sw_right - sw_left
                unit = "m" if PIXELS_PER_METER != 1.0 else "px"
                sw_val = sw_w * cos_correction / PIXELS_PER_METER
                # Draw bracket on footprint panel
                ax_fp.annotate("", xy=(sw_left, sample_row),
                              xytext=(sw_right, sample_row),
                              arrowprops=dict(arrowstyle="<->", color="white",
                                            lw=1.2, shrinkA=0, shrinkB=0))
                ax_fp.text((sw_left + sw_right) / 2, sample_row - 6,
                          f"SW: {sw_val:.0f}{unit}",
                          fontsize=6, color="white", ha="center", va="bottom",
                          bbox=dict(boxstyle="round,pad=0.15", facecolor="gray",
                                  alpha=0.7, edgecolor="none"))

    # Draw connections + width labels for each obstacle
    for ot in obstacle_masks_full_rect:
        full_mask = obstacle_masks_full_rect[ot]
        fp_mask = obstacle_masks_rect.get(ot, np.zeros_like(full_mask))
        color = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))

        if not full_mask.any() or not fp_mask.any():
            continue

        labeled_sil, n_sil = label(full_mask, return_num=True)
        labeled_fp, n_fp = label(fp_mask, return_num=True)
        sil_regions = regionprops(labeled_sil)
        fp_regions = regionprops(labeled_fp)

        for sil_reg in sil_regions:
            sil_cy, sil_cx = sil_reg.centroid
            sil_bbox = sil_reg.bbox  # (min_row, min_col, max_row, max_col)
            sil_w = sil_bbox[3] - sil_bbox[1]
            sil_h = sil_bbox[2] - sil_bbox[0]

            # Label on silhouette panel
            unit = "m" if PIXELS_PER_METER != 1.0 else "px"
            sil_w_val = sil_w * cos_correction / PIXELS_PER_METER
            sil_h_val = sil_h / PIXELS_PER_METER  # height doesn't need correction
            ax_sil.text(sil_cx, sil_bbox[0] - 8, ot,
                       fontsize=7, color="white", ha="center", va="bottom",
                       fontweight="bold",
                       bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85))
            ax_sil.text(sil_cx, sil_bbox[2] + 10,
                       f"{sil_w_val:.0f}×{sil_h_val:.0f}{unit}",
                       fontsize=6, color="white", ha="center", va="top",
                       bbox=dict(boxstyle="round,pad=0.15", facecolor=(0.2, 0.2, 0.2),
                               alpha=0.7, edgecolor="none"))

            # Find closest footprint component
            best_fp_reg = None
            best_dist = float("inf")
            for fp_reg in fp_regions:
                fp_cy, fp_cx = fp_reg.centroid
                d = abs(sil_cx - fp_cx) + abs(sil_cy - fp_cy) * 0.1
                if d < best_dist:
                    best_dist = d
                    best_fp_reg = fp_reg

            if best_fp_reg is None:
                continue

            fp_cy, fp_cx = best_fp_reg.centroid
            fp_bbox = best_fp_reg.bbox
            fp_w = fp_bbox[3] - fp_bbox[1]
            fp_h = fp_bbox[2] - fp_bbox[0]
            fp_w_val = fp_w * cos_correction / PIXELS_PER_METER
            fp_h_val = fp_h / PIXELS_PER_METER  # height doesn't need correction

            # Label on footprint panel
            ax_fp.text(fp_cx, fp_bbox[0] - 8, ot,
                      fontsize=7, color="white", ha="center", va="bottom",
                      fontweight="bold",
                      bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.85))

            # Width bracket below footprint
            bracket_y = fp_bbox[2] + 5
            ax_fp.annotate("", xy=(fp_bbox[1], bracket_y),
                          xytext=(fp_bbox[3], bracket_y),
                          arrowprops=dict(arrowstyle="<->", color="white",
                                        lw=1.5, shrinkA=0, shrinkB=0))
            ax_fp.text(fp_cx, bracket_y + 10,
                      f"w={fp_w_val:.0f}{unit}  h={fp_h_val:.0f}{unit}",
                      fontsize=6, color="white", ha="center", va="top",
                      bbox=dict(boxstyle="round,pad=0.15", facecolor=(0.2, 0.2, 0.2),
                              alpha=0.7, edgecolor="none"))

            # Connection line between panels (silhouette centroid → footprint centroid)
            con = ConnectionPatch(
                xyA=(sil_cx, sil_cy), coordsA=ax_sil.transData,
                xyB=(fp_cx, fp_cy), coordsB=ax_fp.transData,
                color=color, linewidth=2, alpha=0.7,
                arrowstyle="-|>", connectionstyle="arc3,rad=0.15",
                mutation_scale=15)
            fig.add_artist(con)

    # Legend
    map_lh = [
        Patch(facecolor=(0.75, 0.9, 1.0), edgecolor="gray", label="Usable sidewalk"),
        Patch(facecolor=(0.85, 0.85, 0.85), edgecolor="gray", label="Sidewalk (blocked)"),
    ]
    for ot in obstacle_masks_rect:
        is_tree = any(t in ot for t in OBSTACLE_IS_TREE)
        method = "trunk" if is_tree else "base"
        map_lh.append(Patch(facecolor=OBSTACLE_COLORS[ot], label=f"{ot} ({method})"))
    ax_fp.legend(handles=map_lh, loc="upper right", fontsize=8,
                framealpha=0.9, facecolor="white")

    plt.suptitle(f"Silhouette → Footprint Mapping — {sw_key}",
                fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"\n✓ Segment {sw_key} complete.")

    # ── Clean Top-View with Footprint Boxes ──
    fig, (ax_map, ax_legend) = plt.subplots(1, 2, figsize=(10, 14),
                                             gridspec_kw={'width_ratios': [4, 1]})

    # Background: uniform sidewalk color
    sw_color = np.array([0.75, 0.9, 1.0])
    bg_color = np.array([0.15, 0.15, 0.15])
    
    tv_clean = np.ones((*sw.shape, 3), dtype=np.float32) * bg_color
    # Fill the entire sidewalk strip — everything between edge lines is sidewalk
    sw_rows_all = np.where(sw.any(axis=1))[0]
    if len(sw_rows_all) > 0:
        tv_clean[sw_rows_all[0]:sw_rows_all[-1]+1, pad:pad+target_w] = sw_color

    ax_map.imshow(tv_clean)

    # Draw sidewalk boundary lines
    ax_map.axvline(x=pad, color="white", linewidth=1.5, alpha=0.3)
    ax_map.axvline(x=pad + target_w, color="white", linewidth=1.5, alpha=0.3)

    # Draw footprint bounding boxes
    legend_entries = []
    for ot, fp in obstacle_masks_rect.items():
        if not fp.any():
            continue

        color = OBSTACLE_COLORS.get(ot, (1.0, 0.2, 0.2))
        is_tree = any(t in ot for t in OBSTACLE_IS_TREE)
        method = "trunk" if is_tree else "base"

        labeled_fp, n_fp = label(fp, return_num=True)
        for reg in regionprops(labeled_fp):
            min_row, min_col, max_row, max_col = reg.bbox
            box_w = max_col - min_col
            box_h = max_row - min_row

            rect = plt.Rectangle((min_col, min_row), box_w, box_h,
                                  linewidth=2, edgecolor=color,
                                  facecolor=(*color, 0.4))
            ax_map.add_patch(rect)

            # Dimension label inside/below box
            unit = "m" if PIXELS_PER_METER != 1.0 else "px"
            w_val = box_w * cos_correction / PIXELS_PER_METER
            h_val = box_h / PIXELS_PER_METER
            cx_box = (min_col + max_col) / 2.0
            ax_map.text(cx_box, max_row + 8,
                       f"{w_val:.0f}×{h_val:.0f}{unit}",
                       fontsize=6, color="white", ha="center", va="top",
                       bbox=dict(boxstyle="round,pad=0.15",
                                facecolor=(0.2, 0.2, 0.2), alpha=0.8,
                                edgecolor="none"))

        if ot not in [e[0] for e in legend_entries]:
            legend_entries.append((ot, color, method))

    ax_map.set_xlim(pad - 20, pad + target_w + 20)
    sw_rows_all = np.where(sw.any(axis=1))[0]
    if len(sw_rows_all) > 0:
        ax_map.set_ylim(sw_rows_all[-1] + 30, sw_rows_all[0] - 30)
    ax_map.set_title(f"Top View — {sw_key}", fontsize=13, fontweight="bold")
    ax_map.axis("off")

    # ── Side legend panel ──
    ax_legend.axis("off")
    legend_patches = [Patch(facecolor=sw_color, edgecolor="gray", label="Sidewalk")]
    for ot, color, method in legend_entries:
        legend_patches.append(Patch(facecolor=(*color, 0.4), edgecolor=color,
                                     linewidth=2, label=f"{ot} ({method})"))

    ax_legend.legend(handles=legend_patches, loc="center left", fontsize=9,
                     framealpha=0.9, facecolor="white", edgecolor="gray",
                     title="Legend", title_fontsize=10)

    plt.tight_layout()
    plt.show()

## 6. Sidewalk Width Estimation (cm)

Uses the pinhole camera model to convert pixel widths to real-world measurements.

**Camera parameters:** FOV = 90°, pitch = 0°, camera height = 2.5 m

For a pixel at row $v$ in an image of height $H$, the real-world horizontal distance corresponding to $\Delta u$ pixels is:

$$\Delta X = \frac{h \cdot \Delta u}{v - H/2}$$

where $h$ is the camera height. The focal length cancels out (square pixels assumed).

**Bias reduction** — three filters remove unreliable measurements:

1. **Minimum horizon distance** (`MIN_HORIZON_DIST`) — rows near the horizon have tiny $v - H/2$, so even 1-2 px of edge noise gets amplified into huge width values. Rows within this threshold are excluded.
2. **Exclude extrapolated edges** (`EXCLUDE_EXTRAPOLATED`) — when a sidewalk extends beyond the image border, the edge is linearly extrapolated. Far from the camera on the left/right, this extrapolation can overshoot. These rows are flagged and excluded from measurement.
3. **IQR outlier rejection** (`IQR_FACTOR`) — remaining outliers are filtered using the interquartile range fence.

In [ ]:
# ── Bias-reduction parameters ──
MIN_HORIZON_DIST = 20     # ignore rows within this many pixels of the horizon
                          # (small delta_v amplifies edge-detection noise)
EXCLUDE_EXTRAPOLATED = True  # exclude rows with border-extrapolated edges
IQR_FACTOR = 1.5          # IQR multiplier for outlier rejection (1.5 = standard)
EXCLUDE_RANSAC_OUTLIERS = True  # exclude rows flagged as outliers by RANSAC

H_img, W_img = original_image.shape[:2]
cy = H_img / 2   # horizon row (pitch = 0 → image centre)

print(f"Image size : {W_img} × {H_img} px")
print(f"Horizon row: {cy:.0f}")
print(f"Camera height: {CAMERA_HEIGHT_M} m,  HFOV: {HFOV_DEG}°,  pitch: {PITCH_DEG}°")
print(f"Min horizon dist: {MIN_HORIZON_DIST} px,  exclude extrapolated: {EXCLUDE_EXTRAPOLATED},  IQR factor: {IQR_FACTOR}\n")

for sw_key, seg in segments.items():
    sw_mask = seg["sidewalk_mask"]
    left_e, right_e, valid_r, extrap_r, edge_model = find_row_edges(sw_mask, road_mask=combined_road_mask)
    valid_idx = np.where(valid_r)[0]

    # keep only rows below the horizon (ground plane)
    valid_idx = valid_idx[valid_idx > cy]

    # ── Bias fix 1: minimum horizon distance ──
    n_before_horizon_filter = len(valid_idx)
    valid_idx = valid_idx[valid_idx > cy + MIN_HORIZON_DIST]

    # ── Bias fix 2: exclude border-extrapolated rows ──
    n_before_extrap_filter = len(valid_idx)
    if EXCLUDE_EXTRAPOLATED:
        keep = ~extrap_r[valid_idx]
        valid_idx = valid_idx[keep]

    # ── Bias fix 3: exclude RANSAC outlier rows ──
    # These rows had edges corrupted by occlusion (car, pole, etc.)
    # The rectification uses the robust RANSAC line, but for width
    # measurement we want the *observed* edge, so drop outlier rows entirely.
    n_before_ransac_filter = len(valid_idx)
    if EXCLUDE_RANSAC_OUTLIERS and edge_model is not None:
        left_valid_idx_m = edge_model['left_valid_idx']
        right_valid_idx_m = edge_model['right_valid_idx']
        inlier_L = edge_model['inlier_mask_L']
        inlier_R = edge_model['inlier_mask_R']

        # Build sets of inlier rows for each edge independently
        left_inlier_rows = set(left_valid_idx_m[inlier_L]) if len(inlier_L) > 0 else set()
        right_inlier_rows = set(right_valid_idx_m[inlier_R]) if len(inlier_R) > 0 else set()
        # Keep rows that are inliers on BOTH edges
        ransac_inlier_rows = left_inlier_rows & right_inlier_rows

        keep = np.array([r in ransac_inlier_rows for r in valid_idx])
        valid_idx = valid_idx[keep]

    if len(valid_idx) == 0:
        print(f"{sw_key}: no valid rows after filtering – skipped")
        continue

    # ── per-row width in metres ──
    delta_v   = valid_idx - cy                          # pixels below horizon
    width_px_raw = right_e[valid_idx] - left_e[valid_idx]  # sidewalk span (px)

    # Correct for diagonal measurement — horizontal rows cut the sidewalk
    # at an angle when it runs diagonally in the image
    if edge_model is not None:
        a_L_val = edge_model['a_L']
        b_L_val = edge_model['b_L']
        a_R_val = edge_model['a_R']
        b_R_val = edge_model['b_R']

        f_px = W_img / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))  # focal length in pixels

        if abs(a_L_val - a_R_val) > 1e-6:
            vp_y = (b_R_val - b_L_val) / (a_L_val - a_R_val)
            vp_x = a_L_val * vp_y + b_L_val
            # Ground-plane angle from vanishing point offset
            ground_angle = np.arctan((vp_x - W_img / 2.0) / f_px)
        else:
            ground_angle = 0.0

        cos_correction = np.cos(ground_angle)

    else:
        cos_correction = 1.0

    width_px = width_px_raw * cos_correction
    width_m  = CAMERA_HEIGHT_M * width_px / delta_v    # pinhole formula


    width_cm  = width_m * 100

    # ── Bias fix 4: IQR outlier rejection ──
    q1, q3 = np.percentile(width_cm, [25, 75])
    iqr = q3 - q1
    lo_fence = q1 - IQR_FACTOR * iqr
    hi_fence = q3 + IQR_FACTOR * iqr
    inlier = (width_cm >= lo_fence) & (width_cm <= hi_fence)
    n_outliers = (~inlier).sum()

    width_cm_clean = width_cm[inlier]
    valid_idx_clean = valid_idx[inlier]

    # ── statistics (on cleaned data) ──
    med  = np.median(width_cm_clean)
    mean = np.mean(width_cm_clean)
    std  = np.std(width_cm_clean)
    mn, mx = width_cm_clean.min(), width_cm_clean.max()

    print(f"{'='*55}")
    print(f"  Segment : {sw_key}")
    print(f"  Rows (raw below horizon)  : {n_before_horizon_filter}")
    print(f"  Dropped (near horizon)    : {n_before_horizon_filter - n_before_extrap_filter}")
    if EXCLUDE_EXTRAPOLATED:
        print(f"  Dropped (extrapolated)    : {n_before_extrap_filter - n_before_ransac_filter}")
    if EXCLUDE_RANSAC_OUTLIERS and edge_model is not None:
        print(f"  Dropped (RANSAC outliers) : {n_before_ransac_filter - len(valid_idx)}")
    print(f"  Dropped (IQR outliers)    : {n_outliers}")
    if cos_correction < 1.0:
        print(f"  Angle correction          : cos({np.degrees(ground_angle):.1f}°) = {cos_correction:.3f}")
    print(f"  Rows used                 : {len(width_cm_clean)}")
    print(f"  Median  : {med:.1f} cm")
    print(f"  Mean    : {mean:.1f} cm  (σ = {std:.1f} cm)")
    print(f"  Range   : {mn:.1f} – {mx:.1f} cm")
    print(f"{'='*55}\n")

    # ── visualisation ──
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # left: width profile — show all rows (gray) + inliers (blue)
    ax1.plot(valid_idx, width_cm, color="gray", lw=0.5, alpha=0.3, label="All rows")
    ax1.plot(valid_idx_clean, width_cm_clean, "b-", lw=0.8, alpha=0.5, label="Inliers")
    if n_outliers > 0:
        outlier_idx = valid_idx[~inlier]
        outlier_vals = width_cm[~inlier]
        ax1.scatter(outlier_idx, outlier_vals, c="red", s=8, zorder=5, label=f"Outliers ({n_outliers})")
    ax1.axhline(med, color="r", ls="--", label=f"Median {med:.1f} cm")
    ax1.fill_between(valid_idx_clean, mean - std, mean + std,
                     color="blue", alpha=0.08, label=f"±1 σ")
    ax1.axhspan(lo_fence, hi_fence, color="green", alpha=0.04, label="IQR fence")
    ax1.set_xlabel("Image row")
    ax1.set_ylabel("Sidewalk width (cm)")
    ax1.set_title(f"Width profile — {sw_key}")
    ax1.legend(fontsize=7)
    ax1.grid(True, alpha=0.3)

    # right: overlay on original image
    ax2.imshow(original_image, alpha=0.7)
    ax2.axhline(y=cy, color="cyan", linewidth=0.8, linestyle=":", alpha=0.5, label="Horizon")
    ax2.axhline(y=cy + MIN_HORIZON_DIST, color="orange", linewidth=0.8, linestyle=":",
                alpha=0.5, label=f"Min dist ({MIN_HORIZON_DIST}px)")

    norm = plt.Normalize(vmin=max(0, med - 3*std), vmax=med + 3*std)
    cmap = plt.cm.RdYlGn
    # Perpendicular direction to sidewalk travel
    if edge_model is not None:
        a_L_line = edge_model['a_L']
        b_L_line = edge_model['b_L']
        a_R_line = edge_model['a_R']
        b_R_line = edge_model['b_R']

        f_px = W_img / (2.0 * np.tan(np.radians(HFOV_DEG / 2.0)))
        
        if abs(a_L_line - a_R_line) > 1e-6:
            vp_y = (b_R_line - b_L_line) / (a_L_line - a_R_line)
            vp_x = a_L_line * vp_y + b_L_line
            a_avg = (a_L_line + a_R_line) / 2.0
            sign = 1.0 if a_avg > 0 else -1.0
            a_avg_vis = sign * abs((vp_x - W_img / 2.0) / f_px)
        else:
            a_avg_vis = 0.0

    else:
        a_avg_vis = 0.0
        
    perp_dir = np.array([1.0, -a_avg_vis])
    perp_dir /= np.linalg.norm(perp_dir)  # unit vector

    for i, v in enumerate(valid_idx_clean):
        cx = (left_e[v] + right_e[v]) / 2.0
        half_len = (right_e[v] - left_e[v]) * cos_correction / 2.0
        x1 = cx - half_len * perp_dir[0]
        y1 = v  - half_len * perp_dir[1]
        x2 = cx + half_len * perp_dir[0]
        y2 = v  + half_len * perp_dir[1]
        ax2.plot([x1, x2], [y1, y2],
             color=cmap(norm(width_cm_clean[i])), lw=0.6, alpha=0.6)

    # annotate at 25 %, 50 %, 75 % of the valid range
    for frac in (0.25, 0.5, 0.75):
        j = int(len(valid_idx_clean) * frac)
        v = valid_idx_clean[j]
        w = width_cm_clean[j]
        ax2.annotate(
            f"{w:.0f} cm",
            xy=((left_e[v] + right_e[v]) / 2, v),
            fontsize=9, color="white", ha="center",
            bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.7))

    ax2.set_title(f"Width overlay — {sw_key}")
    ax2.legend(loc="upper right", fontsize=7)
    ax2.axis("off")

    plt.suptitle(f"Sidewalk Width Estimation — {sw_key}",
                 fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()